# Исследовательский анализ данных (EDA) — подвыборка SKU-110K

Анализ структуры экспериментального набора данных: распределение числа объектов на изображение, размеров и соотношений сторон ограничивающих рамок, примеры разметки.

Запускать из корня проекта после `python main.py --stage prepare`.

In [ ]:
import glob, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path('..') / 'data' / 'processed'

per_img, areas, ars = {}, {}, {}
for split in ['train', 'val', 'test']:
    p, a, r = [], [], []
    for f in sorted((ROOT / 'labels' / split).glob('*.txt')):
        n = 0
        for line in f.read_text().splitlines():
            _, cx, cy, w, h = map(float, line.split())
            a.append(w * h); r.append(w / max(h, 1e-9)); n += 1
        p.append(n)
    per_img[split], areas[split], ars[split] = p, a, r
    print(f'{split}: изображений {len(p)}, объектов {sum(p)}, '
          f'в среднем {np.mean(p):.1f} на изображение (min {min(p)}, max {max(p)})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].hist(per_img['train'], bins=40, color='#4C72B0', edgecolor='white')
axes[0].axvline(np.mean(per_img['train']), color='#C44E52', ls='--',
                label=f"среднее = {np.mean(per_img['train']):.0f}")
axes[0].set_xlabel('Объектов на изображении'); axes[0].set_ylabel('Изображений')
axes[0].set_title('Плотность сцен'); axes[0].legend(); axes[0].grid(alpha=0.3)

a = np.array(areas['train']) * 100
axes[1].hist(a[a < 2], bins=60, color='#55A868', edgecolor='white')
axes[1].axvline(np.median(a), color='#C44E52', ls='--', label=f'медиана = {np.median(a):.2f}%')
axes[1].set_xlabel('Площадь рамки, % изображения'); axes[1].set_title('Размеры объектов')
axes[1].legend(); axes[1].grid(alpha=0.3)

r = np.array(ars['train']); r = r[r < 4]
axes[2].hist(r, bins=60, color='#8172B3', edgecolor='white')
axes[2].set_xlabel('Ширина/высота рамки'); axes[2].set_title('Соотношения сторон')
axes[2].grid(alpha=0.3)
fig.tight_layout(); plt.show()

In [ ]:
# Примеры изображений с разметкой
from PIL import Image, ImageDraw

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, stem in zip(axes, ['train_00003', 'train_00007']):
    im = Image.open(ROOT / 'images' / 'train' / f'{stem}.jpg').convert('RGB')
    d = ImageDraw.Draw(im); W, H = im.size
    for line in (ROOT / 'labels' / 'train' / f'{stem}.txt').read_text().splitlines():
        _, cx, cy, w, h = map(float, line.split())
        d.rectangle([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H],
                    outline=(220, 40, 40), width=2)
    ax.imshow(im); ax.set_title(stem); ax.axis('off')
plt.tight_layout(); plt.show()

## Выводы

1. Сцены плотноупакованные: в среднем ~146 объектов на изображение (медиана 137, максимум 501) — на порядок плотнее стандартных бенчмарков (COCO: ~7 объектов на изображение).
2. Объекты мелкие: медианная площадь рамки ~0,25% площади изображения; при рабочем разрешении 640 пикселей типичный товар занимает порядка 25–35 пикселей по стороне.
3. Соотношение сторон рамок сосредоточено в диапазоне 0,5–2 (вертикально и горизонтально ориентированные упаковки), экстремальные значения редки.
4. Распределения train/val/test близки, что говорит об однородности сплитов и корректности сравнения моделей на тестовой выборке.